# Model Comparison

Loads saved results from each model and produces side-by-side comparisons.
To add a new model: run its notebook, then add its `results/<model>/tables/` path to `MODELS` below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Register models here as you add them ────────────────────────────────────
MODELS = {
    "xgboost":           Path("../results/xgboost/tables"),
    "linear_regression": Path("../results/linear_regression/tables"),
    # "lightgbm":        Path("../results/lightgbm/tables"),
}

RESULTS_DIR = Path("../results/comparison")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "charts").mkdir(exist_ok=True)
(RESULTS_DIR / "tables").mkdir(exist_ok=True)
print("Registered models:", list(MODELS.keys()))

## 1  Summary Metrics Side-by-Side

In [ ]:
rows = []
for model_name, tables_dir in MODELS.items():
    path = tables_dir / f"{model_name}_summary.csv"
    if not path.exists():
        print(f"WARNING: {path} not found — run {model_name}.ipynb first.")
        continue
    summary = pd.read_csv(path).set_index("metric")["value"]
    summary.name = model_name
    rows.append(summary)

if rows:
    comparison = pd.concat(rows, axis=1)
    print(comparison.to_string())
    comparison.to_csv(RESULTS_DIR / "tables" / "metrics_comparison.csv")
else:
    print("No model results found. Run individual model notebooks first.")

## 2  Cumulative IC Comparison

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for model_name, tables_dir in MODELS.items():
    path = tables_dir / f"{model_name}_ic_monthly.csv"
    if not path.exists():
        continue
    ic = pd.read_csv(path, index_col=0, parse_dates=True)["IC"]
    axes[0].plot(ic.cumsum(), label=model_name)
    axes[1].plot(ic.rolling(12).mean(), label=model_name)

axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].set_title("Cumulative IC")
axes[0].legend()
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].set_title("Rolling 12-Month Mean IC")
axes[1].legend()

fig.suptitle("Model Comparison — Information Coefficient", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "charts" / "ic_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 3  Cumulative L/S Spread Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for model_name, tables_dir in MODELS.items():
    path = tables_dir / f"{model_name}_ls_spread_monthly.csv"
    if not path.exists():
        continue
    ls = pd.read_csv(path, index_col=0, parse_dates=True)["LS_spread"]
    ax.plot(ls.cumsum(), label=model_name)

ax.axhline(0, color="gray", linewidth=0.5)
ax.set_title("Cumulative L/S Rank Spread — All Models")
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS_DIR / "charts" / "ls_spread_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4  Win-Rate Table

For each date, which model had the highest IC?

In [ ]:
ic_frames = {}
for model_name, tables_dir in MODELS.items():
    path = tables_dir / f"{model_name}_ic_monthly.csv"
    if not path.exists():
        continue
    ic_frames[model_name] = pd.read_csv(path, index_col=0, parse_dates=True)["IC"]

if ic_frames:
    ic_all = pd.DataFrame(ic_frames).dropna()
    winner = ic_all.idxmax(axis=1)
    win_rate = winner.value_counts(normalize=True).rename("Win Rate")
    print(win_rate.to_string())
    print()
    mean_ic = ic_all.mean().rename("Mean IC")
    print(mean_ic.to_string())
    
    comparison_table = pd.concat([mean_ic, win_rate], axis=1)
    comparison_table.to_csv(RESULTS_DIR / "tables" / "win_rate.csv")